In [64]:
import os
from dotenv import load_dotenv
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_groq import ChatGroq

In [65]:
# ===== Step 1: Load .env before any getenv
load_dotenv()

True

In [ ]:
# ===== Step 2: Fetch Groq API key from env
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found. Please set it in your .env file")

In [67]:
# ===== Step 3: Load PDF document
pdf_path = "./Project & Thesis Proposal Notice.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

In [68]:
# ===== Step 4: Split text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

In [69]:
# ===== Step 5: Generate embeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [70]:
# ===== Step 6: Setup vectorstore (Chroma)
persist_directory = "Chroma_DB"
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory=persist_directory
)
vectorstore.persist()

In [79]:
# ===== Step 7: Setup Groq LLM
# llm = ChatGroq(
#     model="llama3-8b-8192",
#     temperature=0.3,
#     groq_api_key=GROQ_API_KEY
# )
llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0.3,
    groq_api_key="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"
)

In [80]:
# ===== Step 8: Define prompt template
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are an intelligent assistant. Use the following context to answer the question.

Context:
{context}

Question:
{question}

Answer:"""
)

In [81]:
# ===== Step 9: Create retriever and QA chain
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt_template},
    return_source_documents=True
)

In [86]:
# ===== Step 10: Query example
question = "what is the role of sharmin akter?"
result = qa_chain({"query": question})

In [ ]:
# ===== Step 11: Print results
print("\n=== Answer ===\n")
print(result["result"])

print("\n=== Source Documents Preview ===\n")
for i, doc in enumerate(result["source_documents"]):
    print(f"--- Source Document {i+1} ---")
    print(doc.page_content[:300], "\n")


=== Answer ===

<think>
Okay, let's see. The user is asking about the role of Sharmin Akter based on the provided context. The context is a bit messy with repeated entries, but I need to parse through it carefully.

First, I notice that the context has multiple entries for "Lecturer" followed by phone numbers and names. The key part here is where it mentions "RIB Portal website Sharmin Akter Lecturer 01744960557". That seems to indicate that Sharmin Akter is a lecturer. 

Looking through the repeated sections, each time after the lecturer's name and phone number, there's a line that says "RIB Portal website Sharmin Akter Lecturer..." which might be a note or a tag. Since the word "Lecturer" is right after her name, it's safe to infer that her role is a lecturer. 

I should check if there's any other mention of her role elsewhere in the context. The rest of the entries list other names with phone numbers and IDs, but nothing else about Sharmin Akter. The repetition of the same line sug